<a href="https://colab.research.google.com/github/talhanoor23/my-projects/blob/main/Genetic-Mutation-Detection-System/02_Model_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
df = pd.read_csv('/content/mutation_sequences_kmer.csv')
df.head()

,Chromosome,Start,ReferenceAllele,AlternateAllele,GeneSymbol,label,RefMismatch,Sequence,tokenized
0,chr11,126275389,C,T,FOXRED1,1,False,AAGGTTGGTTTGACCCCTGGTGTCTGCTCCAGGGGCTTCGGCGAAA...,AAGGTT AGGTTG GGTTGG GTTGGT TTGGTT TGGTTT GGTT...
1,chr6,26093008,G,A,HFE,0,False,GGGGTATGTGACTGATGAGAGCCAGGAGCTGAGAAAATCTATTGGG...,GGGGTA GGGTAT GGTATG GTATGT TATGTG ATGTGA TGTG...
2,chr6,26093215,G,T,HFE,1,False,TGCTGTTTTTGTCGTCATCTTGTTCATTGGAATTTTGTTCATAATA...,TGCTGT GCTGTT CTGTTT TGTTTT GTTTTT TTTTTG TTTT...
3,chr2,19989284,T,C,WDR35,1,False,CCTTGTTCCAGGATACACACTGCAGCTTCACGTTATTGGGAATGGA...,CCTTGT CTTGTT TTGTTC TGTTCC GTTCCA TTCCAG TCCA...
4,chr2,19945787,T,C,WDR35,1,False,TATTTGGCTCATTATTTATAGACTGTTAACACTCATTTCTTGTTTT...,TATTTG ATTTGG TTTGGC TTGGCT TGGCTC GGCTCA GCTC...


In [ ]:
# from sklearn.model_selection import train_test_split
# train_val_df, eval_df = train_test_split(df, test_size=0.2, random_state=42)
# train_df, val_df = train_test_split(train_val_df, test_size=0.2, random_state=42)

In [ ]:
!pip install transformers datasets scikit-learn

In [ ]:
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained("zhihan1996/DNA_bert_6")

In [ ]:
from transformers import BertForSequenceClassification
model = BertForSequenceClassification.from_pretrained("zhihan1996/DNA_bert_6", num_labels=2)

pytorch_model.bin:   0%|          | 0.00/359M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/359M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at zhihan1996/DNA_bert_6 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from torch.utils.data import DataLoader
from datasets import Dataset

# Assuming df_clean has your mutation data with 'tokenized' column
def tokenize_function(examples):
    return tokenizer(examples['tokenized'], padding="max_length", truncation=True)

# Convert your dataframe to a Hugging Face dataset
dataset = Dataset.from_pandas(df[['tokenized', 'label']])

# Tokenize the dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Prepare DataLoader for training
train_dataloader = DataLoader(tokenized_dataset, batch_size=8)


Map:   0%|          | 0/260576 [00:00<?, ? examples/s]

In [ ]:
from transformers import Trainer, TrainingArguments

# training_args = TrainingArguments(
#     output_dir="./results",
#     eval_strategy="epoch",
#     save_strategy="epoch", # Added to match eval_strategy for load_best_model_at_end
#     learning_rate=2e-5,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     num_train_epochs=5,
#     weight_decay=0.01,
#     save_total_limit=2,
#     load_best_model_at_end=True
# )
# Training arguments
training_args = TrainingArguments(
    output_dir='./results',          # Output directory
    eval_strategy="epoch",     # Evaluate each epoch
    learning_rate=2e-5,              # Learning rate
    per_device_train_batch_size=8,   # Batch size for training
    num_train_epochs=3,              # Number of epochs
    weight_decay=0.01,               # Weight decay
)

In [ ]:
# Trainer for fine-tuning
trainer = Trainer(
    model=model,                     # The pre-trained model
    args=training_args,              # Training arguments
    train_dataset=tokenized_dataset, # Training dataset
)

# Start fine-tuning
trainer.train()

In [ ]:
model.save_pretrained("./dnabert_mutation_classifier")
tokenizer.save_pretrained("./dnabert_mutation_classifier")

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)